In [1]:
import sys
sys.path.append('..')
from fastai.vision.all import *
from mocatml.utils import *
convert_uuids_to_indices()
from mocatml.data import *
from mocatml.models.conv_rnn import *
from mygrad import sliding_window_view
from tsai.imports import my_setup
from tsai.utils import yaml2dict, dict2attrdict
from fastai.callback.schedule import valley, steep
from fastai.callback.wandb import WandbCallback
import wandb

In [2]:
from fastai.callback.schedule import LRFinder

@patch_to(LRFinder)
def after_fit(self):
    self.learn.opt.zero_grad() # Needed before detaching the optimizer for future fits
    tmp_f = self.path/self.model_dir/self.tmp_p/'_tmp.pth'
    if tmp_f.exists():
        self.learn.load(f'{self.tmp_p}/_tmp', with_opt=True, device='cpu')
        self.tmp_d.cleanup()

In [3]:
my_setup()

os              : Linux-5.10.205-llgrid-x86_64-with-glibc2.35
python          : 3.11.4
tsai            : 0.3.8
fastai          : 2.7.12
fastcore        : 1.5.29
torch           : 2.0.1+cu117
device          : 1 gpu (['Tesla V100-PCIE-32GB'])
cpu cores       : 40
threads per cpu : 2
RAM             : 377.57 GB
GPU memory      : [32.0] GB


In [9]:
config = yaml2dict('./config/convgru/eval.yaml')
config

```json
{ 'data': { 'path': '~/arclab_shared/mc_density_data/comb_am_rp.npy',
            'split_idx': 1},
  'device': 'cuda',
  'horizon': 4,
  'learner': { 'fname': 'learner.pkl',
               'path': 'tmp',
               'wandb': { 'dir': '~/Github/mocat-ml',
                          'download_path': 'wandb/artifacts/density-forecaster',
                          'enabled': False}},
  'lookback': 4,
  'mmap': True,
  'norm': {'mean': None, 'std': None},
  'project': 'mocatml',
  'sel_steps': None,
  'stride': 2,
  'wandb': { 'dir': None,
             'enabled': False,
             'group': None,
             'log_learner': False,
             'mode': 'offline'}}
```

In [5]:
default_device(0 if config.device == 'cpu' else config.device)

device(type='cuda', index=0)

In [6]:
run = wandb.init(dir=ifnone(config.wandb.dir, '../'),
                 project=config.wandb.project, 
                 config=config,
                 group=config.wandb.group,
                 mode=config.wandb.mode, 
                 anonymous='never') if config.wandb.enabled else None
config = dict2attrdict(run.config) if config.wandb.enabled else config
print(config)

{'bs': 32, 'data': {'path': '~/arclab_shared/mc_density_data/comb_am_rp.npy'}, 'device': 'cuda', 'gap': 0, 'horizon': 4, 'lr_max': None, 'lookback': 4, 'mmap': True, 'n_epoch': 1, 'normalize': True, 'num_workers': 0, 'partial_loss': None, 'seed': None, 'save_learner': True, 'sel_steps': None, 'stride': 2, 'tmp_folder': 'tmp', 'wandb': {'dir': None, 'enabled': False, 'log_learner': False, 'mode': 'offline', 'group': None, 'project': 'mocatml'}, 'convgru': {'n_in': 1, 'n_out': 1, 'szs': [16, 64, 96], 'ks': 3, 'rnn_ks': 5, 'blur': False, 'attn': False, 'norm': None, 'strategy': 'zero', 'coord_conv': False, 'debug': False}}


In [10]:
wandb_api = wandb.Api()
if config.learner.wandb.enabled:
    ar = wandb_api.artifact(config.learner.path)
    lp = ar.download(root= Path(config.learner.wandb.dir).expanduser()/\
                            Path(config.learner.wandb.download_path))
    lp = lp/Path(config.learner.fname)
else:
    lp = Path(config.learner.path)/Path(config.learner.fname)
learn = load_learner(lp)
learn

FileNotFoundError: [Errno 2] No such file or directory: 'tmp/learner.pkl'

In [11]:
if config.learner.wandb.enabled:
    run = ar.logged_by()
    config.lookback = run.config['lookback']
    config.horizon = run.config['horizon']
    config.sel_steps = run.config['sel_steps']
    config.stride = run.config['stride']
print(f'lookback: {config.lookback}, horizon: {config.horizon}, sel_steps: {config.sel_steps}, stride: {config.stride}')

lookback: 4, horizon: 4, sel_steps: None, stride: 2


In [12]:
X = np.load(Path(config.data.path).expanduser(), 
               mmap_mode='c' if config.mmap else None)
if config.data.split_idx is not None:
    X = X[learn.splits[config.data.split_idx]]
X.shape

NameError: name 'learn' is not defined

In [7]:
data = np.load(Path(config.data.path).expanduser(), 
               mmap_mode='c' if config.mmap else None)
data = data[:, :config.sel_steps]
data.shape

(100, 2436, 36, 99)

In [8]:
data_sw = np.lib.stride_tricks.sliding_window_view(data, 
                                               config.lookback + config.horizon + config.gap, 
                                               axis=1)[:,::config.stride,:]
samples_per_simulation = data_sw.shape[1]
data_sw = data_sw.transpose(0,1,4,2,3)
data_sw = data_sw.reshape(-1, *data_sw.shape[2:])
data_sw.shape

(121500, 8, 36, 99)